# FinIntel Credit Risk Analysis and Model Selection

This notebook documents the end-to-end analytical workflow for the credit-risk part of FinIntel. It follows the project pipeline from data understanding to preprocessing, feature engineering, comparative model evaluation, and final model selection.

## 1. Setup

The notebook uses the same datasets and model family used by the deployed application so the numbers and conclusions remain aligned with the project codebase.

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

PROJECT_ROOT = Path(r'E:/FraudPulse')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'credit_risk'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_ASSET_DIR = PROJECT_ROOT / 'docs' / 'generated_report_diagrams'
REPORT_ASSET_DIR.mkdir(parents=True, exist_ok=True)

compact_path = PROCESSED_DIR / 'credit_risk_dataset_with_bands.xlsx'
enhanced_path = PROCESSED_DIR / 'Dashboard_Data_Enhanced.csv'
case1_path = RAW_DIR / 'case_study1.xlsx'
case2_path = RAW_DIR / 'case_study2.xlsx'

compact_df = pd.read_excel(compact_path)
enhanced_df = pd.read_csv(enhanced_path)
case1_df = pd.read_excel(case1_path)
case2_df = pd.read_excel(case2_path)

print('case_study1 shape:', case1_df.shape)
print('case_study2 shape:', case2_df.shape)
print('compact dataset shape:', compact_df.shape)
print('enhanced dataset shape:', enhanced_df.shape)

## 2. Dataset Lineage and Structure

In [ ]:
lineage = pd.DataFrame(
    [
        ['case_study1', case1_df.shape[0], case1_df.shape[1], 'Tradeline and bureau summary source'],
        ['case_study2', case2_df.shape[0], case2_df.shape[1], 'Profile, delinquency, enquiry, and approval source'],
        ['credit_risk_dataset_with_bands', compact_df.shape[0], compact_df.shape[1], 'Compact credit modeling dataset'],
        ['Dashboard_Data_Enhanced', enhanced_df.shape[0], enhanced_df.shape[1], 'Enhanced credit modeling and dashboard dataset'],
    ],
    columns=['Dataset', 'Rows', 'Columns', 'Purpose'],
)
lineage

In [ ]:
compact_df.head()

## 3. Exploratory Data Analysis

These charts are designed to be reusable inside the project report.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(data=compact_df, x='Approved_Flag', order=sorted(compact_df['Approved_Flag'].unique()), ax=axes[0], palette='Blues_r')
axes[0].set_title('Approval Category Distribution')
axes[0].set_xlabel('Approved Flag')
axes[0].set_ylabel('Applicants')

sns.histplot(compact_df['NETMONTHLYINCOME'], bins=40, kde=True, ax=axes[1], color='#1f77b4')
axes[1].set_title('Monthly Income Distribution')
axes[1].set_xlabel('Net Monthly Income')

sns.histplot(compact_df['Max_Credit_Amount'], bins=40, kde=True, ax=axes[2], color='#2a9d8f')
axes[2].set_title('Maximum Credit Amount Distribution')
axes[2].set_xlabel('Max Credit Amount')

plt.tight_layout()
plt.savefig(REPORT_ASSET_DIR / 'credit_eda_overview.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=compact_df, x='Approved_Flag', y='NETMONTHLYINCOME', ax=axes[0], palette='Set2')
axes[0].set_title('Income by Approval Category')
axes[0].set_xlabel('Approved Flag')
axes[0].set_ylabel('Net Monthly Income')

sns.boxplot(data=compact_df, x='Approved_Flag', y='Total_TL', ax=axes[1], palette='Set3')
axes[1].set_title('Tradelines by Approval Category')
axes[1].set_xlabel('Approved Flag')
axes[1].set_ylabel('Total Tradelines')

plt.tight_layout()
plt.savefig(REPORT_ASSET_DIR / 'credit_eda_segments.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Feature Engineering Notes

In [ ]:
feature_summary = pd.DataFrame(
    {
        'Compact dataset columns': compact_df.columns,
    }
)
feature_summary.head(13)

In [ ]:
print('Enhanced dataset feature count:', enhanced_df.shape[1])
print('Example engineered fields:')
engineered_cols = [
    'Tot_Missed_Pmnt', 'recent_level_of_deliq', 'CC_enq_L12m', 'PL_enq_L12m',
    'MARITALSTATUS_Married', 'GENDER_F', 'last_prod_enq2_CC', 'Income_Bucket', 'Risk_Profile'
]
print(engineered_cols)

## 5. Preprocessing Helpers

In [ ]:
def evaluate_multiclass_model(name, pipeline, x_train, x_test, y_train, y_test, label_encoder):
    pipeline.fit(x_train, y_train)
    probabilities = pipeline.predict_proba(x_test)
    probabilities = probabilities / probabilities.sum(axis=1, keepdims=True)
    predictions = probabilities.argmax(axis=1)
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Macro_F1': f1_score(y_test, predictions, average='macro'),
        'Weighted_F1': f1_score(y_test, predictions, average='weighted'),
        'Log_Loss': log_loss(y_test, probabilities, labels=list(range(len(label_encoder.classes_)))),
        'Predictions': predictions,
        'Probabilities': probabilities,
        'Pipeline': pipeline,
    }


def build_credit_preprocessor(frame, feature_columns, scale_numeric=False):
    categorical_features = frame[feature_columns].select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    numeric_features = [column for column in feature_columns if column not in categorical_features]
    numeric_transformer = Pipeline([('scaler', StandardScaler())]) if scale_numeric else 'passthrough'
    preprocessor = ColumnTransformer(
        transformers=[
            ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
            ('numeric', numeric_transformer, numeric_features),
        ]
    )
    return preprocessor, categorical_features, numeric_features

## 6. Compact Credit Model Comparison

In [ ]:
compact_target = 'Approved_Flag'
compact_features = [col for col in compact_df.columns if col not in ['Approved_Flag', 'Max_Credit_Amount']]
compact_x = compact_df[compact_features].copy()
compact_label_encoder = LabelEncoder()
compact_y = compact_label_encoder.fit_transform(compact_df[compact_target])

compact_x_train, compact_x_test, compact_y_train, compact_y_test = train_test_split(
    compact_x,
    compact_y,
    test_size=0.2,
    random_state=42,
    stratify=compact_y,
)

compact_preprocessor, compact_categorical, compact_numeric = build_credit_preprocessor(compact_df, compact_features, scale_numeric=False)

compact_models = [
    ('Logistic Regression', LogisticRegression(max_iter=400, multi_class='auto')),
    ('Decision Tree', DecisionTreeClassifier(max_depth=14, random_state=42, class_weight='balanced')),
    ('Random Forest', RandomForestClassifier(n_estimators=350, max_depth=18, min_samples_leaf=2, class_weight='balanced', random_state=42, n_jobs=-1)),
    ('Extra Trees', ExtraTreesClassifier(n_estimators=350, max_depth=18, min_samples_leaf=2, class_weight='balanced', random_state=42, n_jobs=-1)),
]

if XGBClassifier is not None:
    compact_models.append(
        ('XGBoost', XGBClassifier(objective='multi:softprob', num_class=4, n_estimators=450, max_depth=4, learning_rate=0.045, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.5, reg_alpha=0.05, eval_metric='mlogloss', tree_method='hist', random_state=42, n_jobs=-1))
    )

comparison_rows = []
compact_results = {}
for model_name, estimator in compact_models:
    pipeline = Pipeline([('preprocessor', compact_preprocessor), ('classifier', estimator)])
    result = evaluate_multiclass_model(model_name, pipeline, compact_x_train, compact_x_test, compact_y_train, compact_y_test, compact_label_encoder)
    compact_results[model_name] = result
    comparison_rows.append({key: result[key] for key in ['Model', 'Accuracy', 'Macro_F1', 'Weighted_F1', 'Log_Loss']})

if XGBClassifier is not None:
    xgb_model = XGBClassifier(objective='multi:softprob', num_class=4, n_estimators=450, max_depth=4, learning_rate=0.045, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.5, reg_alpha=0.05, eval_metric='mlogloss', tree_method='hist', random_state=42, n_jobs=-1)
    rf_model = RandomForestClassifier(n_estimators=350, max_depth=18, min_samples_leaf=2, class_weight='balanced', random_state=42, n_jobs=-1)
    et_model = ExtraTreesClassifier(n_estimators=350, max_depth=18, min_samples_leaf=2, class_weight='balanced', random_state=42, n_jobs=-1)
    voting = VotingClassifier(estimators=[('xgboost', xgb_model), ('random_forest', rf_model), ('extra_trees', et_model)], voting='soft', weights=[3, 1, 1], n_jobs=-1)
    voting_pipeline = Pipeline([('preprocessor', compact_preprocessor), ('classifier', voting)])
    voting_result = evaluate_multiclass_model('Voting Ensemble', voting_pipeline, compact_x_train, compact_x_test, compact_y_train, compact_y_test, compact_label_encoder)
    compact_results['Voting Ensemble'] = voting_result
    comparison_rows.append({key: voting_result[key] for key in ['Model', 'Accuracy', 'Macro_F1', 'Weighted_F1', 'Log_Loss']})

compact_comparison = pd.DataFrame(comparison_rows).sort_values(['Accuracy', 'Weighted_F1'], ascending=False).reset_index(drop=True)
compact_comparison

In [ ]:
plt.figure(figsize=(10, 5))
plot_df = compact_comparison.melt(id_vars='Model', value_vars=['Accuracy', 'Macro_F1', 'Weighted_F1'], var_name='Metric', value_name='Score')
sns.barplot(data=plot_df, x='Model', y='Score', hue='Metric')
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 1)
plt.title('Compact Credit Model Comparison')
plt.tight_layout()
plt.savefig(REPORT_ASSET_DIR / 'credit_model_comparison_compact.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Enhanced Credit Model Comparison

In [ ]:
enhanced_target = 'Approved_Flag'
enhanced_features = [col for col in enhanced_df.columns if col not in ['Approved_Flag', 'Risk_Profile', 'Income_Bucket']]
enhanced_x = enhanced_df[enhanced_features].copy()
enhanced_label_encoder = LabelEncoder()
enhanced_y = enhanced_label_encoder.fit_transform(enhanced_df[enhanced_target])

enhanced_x_train, enhanced_x_test, enhanced_y_train, enhanced_y_test = train_test_split(
    enhanced_x,
    enhanced_y,
    test_size=0.2,
    random_state=42,
    stratify=enhanced_y,
)

enhanced_preprocessor, enhanced_categorical, enhanced_numeric = build_credit_preprocessor(enhanced_df, enhanced_features, scale_numeric=False)

enhanced_models = [
    ('Logistic Regression', LogisticRegression(max_iter=300, multi_class='auto')),
    ('Decision Tree', DecisionTreeClassifier(max_depth=18, random_state=42, class_weight='balanced')),
    ('Random Forest', RandomForestClassifier(n_estimators=250, max_depth=18, min_samples_leaf=2, class_weight='balanced', random_state=42, n_jobs=-1)),
]
if XGBClassifier is not None:
    enhanced_models.append(
        ('XGBoost', XGBClassifier(objective='multi:softprob', num_class=4, n_estimators=850, max_depth=5, learning_rate=0.035, subsample=0.92, colsample_bytree=0.9, min_child_weight=2, reg_lambda=1.6, reg_alpha=0.04, eval_metric='mlogloss', tree_method='hist', random_state=42, n_jobs=-1))
    )

enhanced_rows = []
enhanced_results = {}
for model_name, estimator in enhanced_models:
    pipeline = Pipeline([('preprocessor', enhanced_preprocessor), ('classifier', estimator)])
    result = evaluate_multiclass_model(model_name, pipeline, enhanced_x_train, enhanced_x_test, enhanced_y_train, enhanced_y_test, enhanced_label_encoder)
    enhanced_results[model_name] = result
    enhanced_rows.append({key: result[key] for key in ['Model', 'Accuracy', 'Macro_F1', 'Weighted_F1', 'Log_Loss']})

enhanced_comparison = pd.DataFrame(enhanced_rows).sort_values(['Accuracy', 'Weighted_F1'], ascending=False).reset_index(drop=True)
enhanced_comparison

In [ ]:
plt.figure(figsize=(9, 5))
plot_df = enhanced_comparison.melt(id_vars='Model', value_vars=['Accuracy', 'Macro_F1', 'Weighted_F1'], var_name='Metric', value_name='Score')
sns.barplot(data=plot_df, x='Model', y='Score', hue='Metric')
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 1)
plt.title('Enhanced Credit Model Comparison')
plt.tight_layout()
plt.savefig(REPORT_ASSET_DIR / 'credit_model_comparison_enhanced.png', dpi=200, bbox_inches='tight')
plt.show()

## 8. Final Credit Model Justification

In [ ]:
final_credit_summary = pd.DataFrame(
    [
        ['Compact Credit', 'Voting Ensemble', compact_results['Voting Ensemble']['Accuracy'] if 'Voting Ensemble' in compact_results else np.nan, compact_results['Voting Ensemble']['Weighted_F1'] if 'Voting Ensemble' in compact_results else np.nan, compact_results['Voting Ensemble']['Log_Loss'] if 'Voting Ensemble' in compact_results else np.nan],
        ['Enhanced Credit', 'XGBoost', enhanced_results['XGBoost']['Accuracy'] if 'XGBoost' in enhanced_results else np.nan, enhanced_results['XGBoost']['Weighted_F1'] if 'XGBoost' in enhanced_results else np.nan, enhanced_results['XGBoost']['Log_Loss'] if 'XGBoost' in enhanced_results else np.nan],
    ],
    columns=['Workflow', 'Selected Model', 'Accuracy', 'Weighted_F1', 'Log_Loss'],
)
final_credit_summary

In [ ]:
best_enhanced = enhanced_results.get('XGBoost')
if best_enhanced is not None:
    cm = confusion_matrix(enhanced_y_test, best_enhanced['Predictions'])
    labels = enhanced_label_encoder.classes_
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title('Enhanced Credit Model Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(REPORT_ASSET_DIR / 'credit_confusion_matrix_enhanced.png', dpi=200, bbox_inches='tight')
    plt.show()
    print(classification_report(enhanced_y_test, best_enhanced['Predictions'], target_names=enhanced_label_encoder.classes_, zero_division=0))

### Conclusion

- The compact credit workflow retains the voting ensemble because it balances stability and multiclass performance on the 13-column borrower dataset.
- The enhanced credit workflow retains XGBoost because it gives the strongest overall accuracy and probability calibration on the engineered 57-column dataset.
- The confusion matrix typically shows that `P2` is easiest to learn while `P3` remains the most difficult approval segment, which is consistent with the deployed project findings.